In [1]:
import torch
import time
import numpy as np
import pandas as pd
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR100
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from tensorflow.keras.datasets import cifar100
from torch.utils.data import TensorDataset, DataLoader,Dataset
from sklearn.model_selection import train_test_split

/Users/susiewonfor/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
data = pd.read_csv("sudoku_cluewise.csv")


(4000000, 3)

In [7]:
data

,quizzes,solutions,clue_numbers
0,3274568919183725464659817325328641797915236848...,3274568919183725464659817325328641797915236848...,80
1,1538972649876241532645139876327518498493625705...,1538972649876241532645139876327518498493625715...,80
2,6851374293174928569428651738295167341567432984...,6851374293174928569428651738295167341567432984...,80
3,8396245714207513981579382643941627852618759435...,8396245714267513981579382643941627852618759435...,80
4,3168275944593617288724951636872543192459138769...,3168275944593617288724951636872543192459138769...,80
...,...,...,...
3999995,0005060000020000500600000170000000002300000000...,1475968323821746595693284177169853242347615989...,17
3999996,0040020000000900000060000001000400000900004004...,9846327157514983623261759481325476896982134574...,17
3999997,0000050000000001000902000000450000000600003000...,1728654396583491274932176858459312762675843913...,17
3999998,0090000007300000055000000006000080000050000000...,8496521377314892655623719486271385944952678133...,17


In [6]:
data.shape

(4000000, 3)

In [5]:
data['puzzle_array'] = data['quizzes'].apply(lambda x: np.array(list(x), dtype=int).reshape(9, 9))
data['solution_array'] = data['solutions'].apply(lambda x: np.array(list(x), dtype=int).reshape(9, 9))

In [9]:
easydat1 = data[(data["clue_numbers"] >= 70) & (data["clue_numbers"] <= 80)]
easydat2 = data[(data["clue_numbers"] >= 60) & (data["clue_numbers"] < 75)]
easydat3 = data[(data["clue_numbers"] >= 50) & (data["clue_numbers"] < 65)]
meddat1 = data[(data["clue_numbers"] >= 40) & (data["clue_numbers"] < 55)]
meddat2 = data[(data["clue_numbers"] >= 30) & (data["clue_numbers"] < 45)]
meddat3 = data[(data["clue_numbers"] >= 20) & (data["clue_numbers"] < 35)]
harddat = data[(data["clue_numbers"] >= 17) & (data["clue_numbers"] < 25)]

datasets = [
    easydat1,
    easydat2,
    easydat3,
    meddat1,
    meddat2,
    meddat3,
    harddat
]

In [ ]:
samplesize = 90000

for i, df in enumerate(datasets):
    datasets[i] = df.sample(n=samplesize, random_state=42)
easydat1 = easydat1.sample(n = samplesize,random_state=42)

In [10]:
bch_size = 64
train_set = []
test_set = []

for x in datasets:
    train, test = train_test_split(x, test_size=0.2, random_state=42)

    X_train = np.stack(train["puzzle_array"].to_numpy())
    Y_train = np.stack(train["solution_array"].to_numpy())
    X_test = np.stack(test["puzzle_array"].to_numpy())
    Y_test = np.stack(test["solution_array"].to_numpy())

    # Convert to torch tensors and normalize
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1) / 9.0  # (N, 1, 9, 9)
    Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32).unsqueeze(1) / 9.0
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1) / 9.0
    Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32).unsqueeze(1) / 9.0

    # Wrap in TensorDataset
    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=bch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=bch_size, shuffle=False)

    train_set.append(train_loader)
    test_set.append(test_loader)

In [11]:
from torch import nn

class SudokuCNNClassifier(nn.Module):
    def __init__(self):
        super(SudokuCNNClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv6 = nn.Conv2d(512, 10, kernel_size=1)  # 10 classes per cell
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout2d(p=0.3)

        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(128)
        self.bn3 = nn.BatchNorm2d(256)
        self.bn4 = nn.BatchNorm2d(512)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.dropout(self.bn1(x))

        x = self.relu(self.conv2(x))
        x = self.dropout(self.bn2(x))

        x = self.relu(self.conv3(x))
        x = self.dropout(self.bn3(x))

        x = self.relu(self.conv4(x))
        x = self.dropout(self.bn4(x))

        x = self.relu(self.conv5(x))
        x = self.dropout(self.bn4(x))
        return self.conv6(x)  # shape (batch, 10, 9, 9)

In [12]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


In [82]:
def sudoku_penalty(preds):
    """
    Penalize duplicate digits in Sudoku rows and columns.
    preds: (B, 9, 9) LongTensor with values 0–8
    """
    penalty = 0.0
    batch_size = preds.size(0)

    for b in range(batch_size):
        grid = preds[b]  # (9, 9)
        for i in range(9):
            # Count duplicates in rows and columns
            row_counts = torch.bincount(grid[i], minlength=9)
            col_counts = torch.bincount(grid[:, i], minlength=9)

            # Penalize entries that appear more than once
            penalty += (row_counts > 1).float().sum()
            penalty += (col_counts > 1).float().sum()

    # Average over batch
    return penalty / batch_size

In [13]:
import torch.nn.functional as F

model = SudokuCNNClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
criterion = torch.nn.CrossEntropyLoss()

n_epochs = 3
set_count = 0

for train_loader in train_set:
    set_count += 1
    print(f"\nTraining set: {set_count}")

    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.long().to(device).squeeze(1)
            
            mask = (X_batch == 0).squeeze(1) 

            optimizer.zero_grad()
            out = model(X_batch)
            # preds = out.argmax(dim=1)
            loss_masked = criterion(out.permute(0, 2, 3, 1)[mask], Y_batch[mask])
            loss_full = criterion(out.permute(0, 2, 3, 1).reshape(-1, 10), Y_batch.reshape(-1))
            loss = 0.9 * loss_masked + 0.1 * loss_full
            lambda_reg = 1e-3  # good starting value
            # loss += lambda_reg * sudoku_penalty(preds)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1}, Loss: {avg_loss:.4f}")


Training set: 1
Epoch 1, Loss: 0.0490


KeyboardInterrupt: 

In [85]:
model.eval()  # disable dropout, batchnorm updates
with torch.no_grad():  # no gradients needed
    for X_batch, Y_batch in test_loader:
        X_batch = X_batch.to(device)                    # (B, 1, 9, 9)
    Y_batch = Y_batch.long().to(device).squeeze(1)  # (B, 9, 9)

# Create mask from current batch
    mask = (X_batch == 0).squeeze(1)               # (B, 9, 9)

# Forward pass
    out = model(X_batch)                            # (B, 10, 9, 9)
    preds = out.argmax(dim=1)                       # (B, 9, 9)

# Keep clues fixed
    solved = X_batch.squeeze(1).long().clone()      # (B, 9, 9)
    solved[mask] = preds[mask]                      # works now


In [86]:
model.eval()
set_count = 0

total_correct_cells = 0
total_cells = 0
total_correct_puzzles = 0
total_puzzles = 0

with torch.no_grad():
    for test_loader in test_set:
        set_count += 1
        print(f"\n🧩 Test set {set_count}")

        correct_cells = 0
        total_cells_set = 0
        correct_puzzles = 0
        total_puzzles_set = 0

        for X_batch, Y_batch in test_loader:
            X_batch = X_batch.to(device)                  # (B, 1, 9, 9)
            Y_batch = Y_batch.long().to(device).squeeze(1)  # (B, 9, 9)

            out = model(X_batch)                          # (B, 10, 9, 9)
            preds = out.argmax(dim=1)                     # (B, 9, 9)

            # Cell-level accuracy
            correct_cells += (preds == Y_batch).sum().item()
            total_cells_set += torch.numel(Y_batch)

            # Puzzle-level accuracy
            correct_puzzles += (preds == Y_batch).all(dim=(1, 2)).sum().item()
            total_puzzles_set += Y_batch.shape[0]

        cell_acc = correct_cells / total_cells_set * 100
        puzzle_acc = correct_puzzles / total_puzzles_set * 100

        print(f"Cell accuracy: {cell_acc:.2f}% | Full puzzle accuracy: {puzzle_acc:.2f}%")

        # accumulate totals
        total_correct_cells += correct_cells
        total_cells += total_cells_set
        total_correct_puzzles += correct_puzzles
        total_puzzles += total_puzzles_set

overall_cell_acc = total_correct_cells / total_cells * 100
overall_puzzle_acc = total_correct_puzzles / total_puzzles * 100

print(f"\n✅ Overall cell accuracy: {overall_cell_acc:.2f}%")
print(f"🏆 Overall full puzzle accuracy: {overall_puzzle_acc:.2f}%")


🧩 Test set 1
Cell accuracy: 99.26% | Full puzzle accuracy: 57.30%

🧩 Test set 2
Cell accuracy: 98.30% | Full puzzle accuracy: 25.60%

🧩 Test set 3
Cell accuracy: 96.83% | Full puzzle accuracy: 5.50%

🧩 Test set 4
Cell accuracy: 95.45% | Full puzzle accuracy: 1.50%

🧩 Test set 5
Cell accuracy: 94.01% | Full puzzle accuracy: 0.20%

🧩 Test set 6
Cell accuracy: 92.56% | Full puzzle accuracy: 0.00%

🧩 Test set 7
Cell accuracy: 91.74% | Full puzzle accuracy: 0.00%

✅ Overall cell accuracy: 95.45%
🏆 Overall full puzzle accuracy: 12.87%


In [64]:
import torch

model.eval()
set_count = 0

total_correct_cells = 0
total_cells = 0
total_correct_puzzles = 0
total_puzzles = 0

with torch.no_grad():
    for test_loader in test_set:
        set_count += 1
        print(f"\n🧩 Test set {set_count}")

        correct_cells = 0
        total_cells_set = 0
        correct_puzzles = 0
        total_puzzles_set = 0

        for X_batch, Y_batch in test_loader:
            X_batch = X_batch.to(device)                    # (B, 1, 9, 9)
            Y_batch = Y_batch.long().to(device).squeeze(1)  # (B, 9, 9)

            # Create mask from current batch (empty cells)
            mask = (X_batch == 0).squeeze(1)                # (B, 9, 9)

            # Forward pass
            out = model(X_batch)                            # (B, 10, 9, 9)
            preds = out.argmax(dim=1)                       # (B, 9, 9)

            # Keep clues fixed
            solved = X_batch.squeeze(1).long().clone()
            solved[mask] = preds[mask]

            # ✅ Cell accuracy only on masked cells
            correct_cells += (preds[mask] == Y_batch[mask]).sum().item()
            total_cells_set += mask.sum().item()

            # ✅ Full puzzle accuracy (all masked cells correct)
            for i in range(Y_batch.shape[0]):
                if (preds[i][mask[i]] == Y_batch[i][mask[i]]).all():
                    correct_puzzles += 1
            total_puzzles_set += Y_batch.shape[0]

        cell_acc = correct_cells / total_cells_set * 100
        puzzle_acc = correct_puzzles / total_puzzles_set * 100

        print(f"Masked cell accuracy: {cell_acc:.2f}% | Full puzzle accuracy: {puzzle_acc:.2f}%")

        total_correct_cells += correct_cells
        total_cells += total_cells_set
        total_correct_puzzles += correct_puzzles
        total_puzzles += total_puzzles_set

overall_cell_acc = total_correct_cells / total_cells * 100
overall_puzzle_acc = total_correct_puzzles / total_puzzles * 100

print(f"\n✅ Overall masked cell accuracy: {overall_cell_acc:.2f}%")
print(f"🏆 Overall full puzzle accuracy: {overall_puzzle_acc:.2f}%")



🧩 Test set 1
Masked cell accuracy: 95.39% | Full puzzle accuracy: 77.05%

🧩 Test set 2
Masked cell accuracy: 93.82% | Full puzzle accuracy: 46.52%

🧩 Test set 3
Masked cell accuracy: 91.96% | Full puzzle accuracy: 19.65%

🧩 Test set 4
Masked cell accuracy: 90.48% | Full puzzle accuracy: 4.98%

🧩 Test set 5
Masked cell accuracy: 89.39% | Full puzzle accuracy: 1.10%

🧩 Test set 6
Masked cell accuracy: 88.96% | Full puzzle accuracy: 0.07%

🧩 Test set 7
Masked cell accuracy: 88.92% | Full puzzle accuracy: 0.00%

✅ Overall masked cell accuracy: 90.00%
🏆 Overall full puzzle accuracy: 21.34%
